## TuneBuddy system flow

Load in **AUDIO** into UserData
- Computes pitches with `PitchDetector` -> creates `PitchData`
- Computes notes with `NoteDetector` -> creates `NoteData`

Load in **MIDI** into MidiData
- Creates `message_dict` which can be read by MidiSynth
- Creates `NoteData`

Finally, compare AUDIO.NoteData to MIDI.NoteData using `StringEditor`

In [ ]:
# autoreload when modules edited
%load_ext autoreload
%autoreload 2

In [6]:
import sys
sys.path.append('..')

from app_logic.midi.MidiData import MidiData
from app_logic.user.ds.UserData import UserData
# from app_logic.user.AudioRecorder import AudioRecorder
# from app_logic.user.AudioPlayer import AudioPlayer

from algorithms.PitchDetector import PitchDetector
from algorithms.NoteDetector import NoteDetector
from algorithms.StringEditor import StringEditor
from algorithms.Config import Config

In [8]:
# test pitch detection accuracy
AUDIO_PATH = "../resources/recordings/bach_fugue_recording.mp3"
MIDI_PATH = "../resources/scores/fugue.mid"
SF_PATH = "../resources/MuseScore_General.sf3"

config = {
    'sr': 44100,    # sample rate
    'w1': 1024 * 8,  # frame size
    'h1': 128,       # hop size
    'fmin': 196.0,
    'fmax': 3000.0,
    'tuning': 440.0,
    'unv_thresh': 0.8, # if unvoiced_prob > unv_thresh, consider the frame unvoiced

    # --- NOTE DETECTION PARAMETERS ---
    'w2': 30, # frame size
    'h2': 27, # hop size
    'pitch_thresh': 0.75,
    'slope_thresh': 1.5,
    'unv_ratio': 0.5, # proportion of unvoiced pitches in a window to consider the window unvoiced

    # --- STRING EDIT PARAMETERS ---
    'ins_cost': 1.5,
    'del_cost': 2,
    'sub_cost': 1,
    'tolerance': 1,
    # tiger-mom parameter
    'tiger_level': 1
}
config = Config(**config)

# create MIDI data
mid = MidiData(MIDI_PATH)

# create USER data, computing pitches and notes
pd, nd, se = PitchDetector(config), NoteDetector(config), StringEditor(config)
us = UserData(pd, nd)
us.load_audio(AUDIO_PATH)

mid.resize(new_length=us.audio_data.get_length())

align = se.string_edit(
    us.note_data, 
    mid.note_data
)

Loading score file: ../resources/scores/fugue.mid (ext: .mid)
Handling MIDI file...
Starting pitch detection...
Processing frame 4450/4450
Done!
Starting note detection...
Note detection: Done!
Starting string editing...
Done!


### Visualize the results

In [9]:
%gui qt

import sys
from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import QCoreApplication

sys.path.append('../')
from ui.ScorePlot import RunScorePlot

if __name__ == '__main__':
    if not QCoreApplication.instance():
        app = QApplication(sys.argv)
    else:
        app = QCoreApplication.instance()

    vis = RunScorePlot(user_data=us, midi_data=mid, alignment=align, app=app)

Loading MIDI data into ScorePlot...
Loading UserData into ScorePlot...
Plotting alignment...
